# Notebook 04 — SHAP Explainability

Deep-dive into model explanations using SHAP values.
Run after `03_model_training.ipynb`.

In [ ]:
!pip install shap xgboost scikit-learn pandas numpy matplotlib -q

import pickle, pandas as pd, numpy as np, shap
import matplotlib.pyplot as plt
plt.style.use('dark_background')

model         = pickle.load(open('ml/model_artifacts/xgb_model.pkl',    'rb'))
feature_names = pickle.load(open('ml/model_artifacts/feature_names.pkl','rb'))
df            = pd.read_csv('data/processed/churn_processed.csv')

X = df.drop(columns=['Churn'])
y = df['Churn']
print(f'Data loaded: {X.shape}')

In [ ]:
# ── Compute SHAP values ───────────────────────────────────────────────────
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)
print(f'SHAP values shape: {np.array(shap_values).shape}')

In [ ]:
# ── Summary plot (beeswarm) ───────────────────────────────────────────────
shap.summary_plot(shap_values, X, feature_names=feature_names, show=False)
plt.tight_layout(); plt.show()
print('Each dot = one customer. Color = feature value. X position = impact on prediction.')

In [ ]:
# ── Bar summary (global importance) ──────────────────────────────────────
shap.summary_plot(shap_values, X, feature_names=feature_names, plot_type='bar', show=False)
plt.title('Mean |SHAP| — Global Feature Importance')
plt.tight_layout(); plt.show()

In [ ]:
# ── Single customer explanation (waterfall) ───────────────────────────────
# Pick a high-risk customer
y_proba = model.predict_proba(X)[:,1]
high_risk_idx = np.argmax(y_proba)
print(f'Highest risk customer index: {high_risk_idx}, probability: {y_proba[high_risk_idx]:.3f}')

explanation = shap.Explanation(
    values       = shap_values[high_risk_idx],
    base_values  = explainer.expected_value,
    data         = X.iloc[high_risk_idx].values,
    feature_names= feature_names
)
shap.plots.waterfall(explanation, show=False)
plt.tight_layout(); plt.show()

In [ ]:
# ── SHAP dependence plot: Contract vs Churn ───────────────────────────────
contract_idx = feature_names.index('Contract')
shap.dependence_plot(
    contract_idx, shap_values, X,
    feature_names=feature_names, show=False
)
plt.title('SHAP Dependence: Contract Type')
plt.tight_layout(); plt.show()

In [ ]:
# ── Top 3 factors for 5 random customers ──────────────────────────────────
sample_indices = np.random.choice(len(X), 5, replace=False)

for idx in sample_indices:
    prob   = y_proba[idx]
    sv     = shap_values[idx]
    top3   = sorted(zip(feature_names, sv), key=lambda x: abs(x[1]), reverse=True)[:3]
    risk   = 'HIGH' if prob > 0.65 else 'MEDIUM' if prob > 0.35 else 'LOW'
    print(f'Customer {idx:4d} | Prob: {prob:.3f} | Risk: {risk}')
    for feat, val in top3:
        direction = '↑ increases' if val > 0 else '↓ decreases'
        print(f'  {direction} churn: {feat:30s}  SHAP={val:+.4f}')
    print()